# Phase 2 Demo: Middleware, Safety Rails, and Agent-Tool Wiring

This notebook demonstrates the Phase 2 implementation:
- Middleware stack (limits, PII, HITL, retry, fallback, summarization)
- Tool binding per department
- HITL checkpoints and simulation
- End-to-end workflow execution

**Phase 2 Goals:**
1. Make the agent core production-ready
2. Wire robust middleware stack
3. Define approval checkpoints
4. Bind tools to departments

## 1. Imports & Setup

In [32]:
import sys
sys.path.insert(0, '..')

import json
from datetime import datetime
from pprint import pprint

# Middleware components
from middleware.chain import MiddlewareChain, get_middleware_chain, ExecutionContext
from middleware.handlers.limits_handler import LimitsHandler
from middleware.handlers.pii_handler import PIIHandler
from middleware.handlers.hitl_handler import HITLHandler
from middleware.handlers.style_injector import StyleInjectorHandler
from middleware.hitl_simulator import HITLSimulator, SimulatorMode
from dotenv import load_dotenv

load_dotenv()

# Tool system
from agents.tool_binder import ToolBinder, get_tool_binder

# Agent nodes
from agents.nodes import (
    researcher_node,
    writer_node,
    editor_node,
    diagrammer_node,
)

# Workflow
from workflows.newsletter_graph import (
    NewsletterWorkflow,
    create_newsletter_graph,
    create_initial_state,
)

print("All imports successful!")

All imports successful!


## 2. Configuration Loading

Load policy.yaml and style profile configurations.

In [33]:
import yaml
from pathlib import Path

# Load policy configuration
policy_path = Path('../config/policy.yaml')
with open(policy_path) as f:
    policy = yaml.safe_load(f)

print("=== Policy Configuration ===")
print(f"\nLimits:")
print(f"  - Max model calls: {policy['limits']['max_model_calls_per_run']}")
print(f"  - Max tool calls: {policy['limits']['max_tool_calls_per_run']}")
print(f"  - Max depth: {policy['limits']['max_depth_per_run']}")

print(f"\nHITL Settings:")
print(f"  - Gate unknown domains: {policy['hitl']['gate_web_fetch_unknown_domain']}")
print(f"  - Allowed domains: {len(policy['hitl']['allow_domains'])} domains")

print(f"\nFallback:")
print(f"  - Primary: {policy['fallback']['primary_model']}")
print(f"  - Secondary: {policy['fallback']['secondary_model']}")

=== Policy Configuration ===

Limits:
  - Max model calls: 30
  - Max tool calls: 50
  - Max depth: 8

HITL Settings:
  - Gate unknown domains: True
  - Allowed domains: 36 domains

Fallback:
  - Primary: gpt-4o
  - Secondary: claude-sonnet-4-5-20250929


In [34]:
# Load style profile
style_path = Path('../config/style_profiles/neutral_concise.json')
with open(style_path) as f:
    style_profile = json.load(f)

print("=== Style Profile ===")
print(f"Name: {style_profile['name']}")
print(f"Voice: {style_profile['voice']}")
print(f"Sentence Length: ({style_profile['readability']['sentence_length_min']}, {style_profile['readability']['sentence_length_max']})")
print(f"Structure: {style_profile['structure']['sections']}")

=== Style Profile ===
Name: Neutral Concise
Voice: neutral, confident, helpful
Sentence Length: (12, 20)
Structure: ['hook', 'summary', 'sections', 'key-takeaways', 'CTA']


## 3. Middleware Stack Demo

Demonstrate each middleware handler in isolation.

### 3.1 Limits Handler

In [35]:
# Test limits handler
limits_config = {
    "max_model_calls_per_run": 5,
    "max_tool_calls_per_run": 10,
    "max_depth_per_run": 3,
}

limits_handler = LimitsHandler(limits_config)

# Test within limits
state_ok = {"model_calls": 2, "tool_calls": 5, "depth": 1}
result = limits_handler.pre_process(state_ok)
print("Within limits - PASSED")

# Test exceeding limits
state_exceeded = {"model_calls": 10, "tool_calls": 5, "depth": 1}
try:
    limits_handler.pre_process(state_exceeded)
except ValueError as e:
    print(f"Exceeding limits - BLOCKED: {e}")

Within limits - PASSED
Exceeding limits - BLOCKED: Model call limit exceeded: 10/5


### 3.2 PII Handler

In [36]:
# Test PII redaction
import copy

pii_handler = PIIHandler({})  # Pass empty config

# Text with PII
state_with_pii = {
    "content": """
    Contact our team at support@company.com or call 555-123-4567.
    For billing, reach out to billing@company.com.
    SSN for verification: 123-45-6789
    """
}
state_with_pii_org = copy.deepcopy(state_with_pii)

result = pii_handler.post_process(state_with_pii)

print("=== PII Redaction Demo ===")
print("\nOriginal:")
print(state_with_pii_org["content"])
print("\nRedacted:")
print(result["content"])

=== PII Redaction Demo ===

Original:

    Contact our team at support@company.com or call 555-123-4567.
    For billing, reach out to billing@company.com.
    SSN for verification: 123-45-6789
    

Redacted:

    Contact our team at [EMAIL_REDACTED] or call 555-123-4567.
    For billing, reach out to [EMAIL_REDACTED].
    SSN for verification: [SSN_REDACTED]
    


### 3.3 HITL Handler

In [37]:
# Test HITL gating
hitl_config = {
    "gate_web_fetch_unknown_domain": True,
    "allow_domains": ["kafka.apache.org", "confluent.io", "docs.langchain.com"],
}

hitl_handler = HITLHandler(hitl_config)

# Test trusted domain
state_trusted = {
    "tool_name": "web.fetch",
    "tool_args": {"url": "https://kafka.apache.org/documentation"},
}
result_trusted = hitl_handler.pre_process(state_trusted)
print(f"Trusted domain (kafka.apache.org): HITL required = {result_trusted['hitl_required']}")

# Test unknown domain
state_unknown = {
    "tool_name": "web.fetch",
    "tool_args": {"url": "https://unknown-blog.com/article"},
}
result_unknown = hitl_handler.pre_process(state_unknown)
print(f"Unknown domain (unknown-blog.com): HITL required = {result_unknown['hitl_required']}")

Trusted domain (kafka.apache.org): HITL required = False
Unknown domain (unknown-blog.com): HITL required = True


### 3.4 Style Profile Injector

In [38]:
# Test style injection
style_injector = StyleInjectorHandler({})  # Pass empty config

# With style profile
state_with_style = {
    "style_profile": style_profile,
}
result = style_injector.pre_process(state_with_style)
print(f"With style profile: Using '{result['style_profile']['name']}'")

# Without style profile
state_no_style = {}
result = style_injector.pre_process(state_no_style)
print(f"Without style profile: Using default '{result.get('style_profile_used', 'N/A')}'")

With style profile: Using 'Neutral Concise'
Without style profile: Using default 'default'


## 4. Tool Binding per Department

Demonstrate tool access control by department.

In [39]:
# Get the tool binder
tool_binder = ToolBinder()

print("=== Tool Binding by Department ===")

departments = ["researcher", "writer", "editor", "diagrammer"]

for dept in departments:
    allowed = tool_binder.get_allowed_tools(dept)
    denied = tool_binder.get_denied_tools(dept)
    print(f"\n{dept.upper()}:")
    print(f"  Allowed: {allowed}")
    print(f"  Denied: {denied}")

=== Tool Binding by Department ===

RESEARCHER:
  Allowed: ['web.search', 'web.fetch', 'local.search', 'citation.builder']
  Denied: []

WRITER:
  Allowed: ['seo.readability']
  Denied: ['web.fetch', 'web.search']

EDITOR:
  Allowed: ['seo.readability', 'plagiarism.scan', 'citation.builder']
  Denied: ['web.fetch']

DIAGRAMMER:
  Allowed: ['drawio.generate', 'drawio.export', 'web.search']
  Denied: ['web.fetch']


In [40]:
# Test tool access control
print("=== Access Control Tests ===")

test_cases = [
    ("researcher", "web.search", True),
    ("researcher", "web.fetch", True),
    ("writer", "web.fetch", False),
    ("writer", "seo.readability", True),
    ("editor", "web.fetch", False),
    ("editor", "seo.readability", True),
    ("diagrammer", "drawio.generate", True),
    ("diagrammer", "web.search", True),
]

for agent, tool, expected in test_cases:
    result = tool_binder.is_tool_allowed(agent, tool)
    status = "PASS" if result == expected else "FAIL"
    symbol = "" if result else ""
    print(f"  {agent} -> {tool}: {symbol} ({status})")

=== Access Control Tests ===
  researcher -> web.search:  (PASS)
  researcher -> web.fetch:  (PASS)
  writer -> web.fetch:  (PASS)
  writer -> seo.readability:  (PASS)
  editor -> web.fetch:  (PASS)
  editor -> seo.readability:  (PASS)
  diagrammer -> drawio.generate:  (PASS)
  diagrammer -> web.search:  (PASS)


## 5. HITL Simulator Demo

Demonstrate HITL simulation modes.

In [41]:
# Create simulators for different modes
auto_approve = HITLSimulator(mode=SimulatorMode.AUTO_APPROVE)
auto_deny = HITLSimulator(mode=SimulatorMode.AUTO_DENY)
policy_mode = HITLSimulator(
    mode=SimulatorMode.POLICY,
    allow_domains=["kafka.apache.org", "confluent.io"],
)

# For POLICY mode, the simulator checks the 'risk.domain' field
# This is typically set by the HITLHandler when it detects a risky operation

# Request for trusted domain - includes risk info with domain
request_trusted = {
    "checkpoint_id": "CP_001",
    "agent": "researcher",
    "tool": "web.fetch",
    "args": {"url": "https://kafka.apache.org/docs"},
    "risk": {"domain": "kafka.apache.org", "domain_trust": "unknown"},  # Domain in allow list
}

# Request for untrusted domain
request_untrusted = {
    "checkpoint_id": "CP_002", 
    "agent": "researcher",
    "tool": "web.fetch",
    "args": {"url": "https://random-blog.com/article"},
    "risk": {"domain": "random-blog.com", "domain_trust": "unknown"},  # Domain NOT in allow list
}

print("Trusted Domain (kafka.apache.org - in allow_domains list):")
result = auto_approve.process(request_trusted)
print(f"  AUTO_APPROVE: {result['decision']} - {result['rationale']}")
result = auto_deny.process(request_trusted)
print(f"  AUTO_DENY: {result['decision']} - {result['rationale']}")
result = policy_mode.process(request_trusted)
print(f"  POLICY: {result['decision']} - {result['rationale']}")

print("\nUntrusted Domain (random-blog.com - NOT in allow_domains list):")
result = auto_approve.process(request_untrusted)
print(f"  AUTO_APPROVE: {result['decision']} - {result['rationale']}")
result = auto_deny.process(request_untrusted)
print(f"  AUTO_DENY: {result['decision']} - {result['rationale']}")
result = policy_mode.process(request_untrusted)
print(f"  POLICY: {result['decision']} - {result['rationale']}")

Trusted Domain (kafka.apache.org - in allow_domains list):
  AUTO_APPROVE: approve - Auto-approved by simulator
  AUTO_DENY: deny - Auto-denied by simulator
  POLICY: approve - Domain kafka.apache.org is in allow list

Untrusted Domain (random-blog.com - NOT in allow_domains list):
  AUTO_APPROVE: approve - Auto-approved by simulator
  AUTO_DENY: deny - Auto-denied by simulator
  POLICY: deny - Domain random-blog.com is not in allow list


## 6. Agent Nodes Demo

Demonstrate individual agent node execution.

In [42]:
# Create initial state for testing
test_state = create_initial_state(
    topic="Apache Kafka Fundamentals",
    brief="Write a comprehensive guide about Kafka for beginners",
    requirements=["core concepts", "architecture", "use cases"],
    style_profile=style_profile,
    max_iterations=3,
)

print("=== Initial State ===")
print(f"Run ID: {test_state['run_id']}")
print(f"Topic: {test_state['tasks'][0]['topic']}")
print(f"Current Agent: {test_state['current_agent']}")
print(f"Workflow Status: {test_state['workflow_status']}")

=== Initial State ===
Run ID: run_e82a2bb3cd79
Topic: Apache Kafka Fundamentals
Current Agent: researcher
Workflow Status: running


In [43]:
# Execute researcher node
print("=== Researcher Node Execution ===")
researcher_result = researcher_node(test_state)

print(f"Facts gathered: {len(researcher_result.get('facts', []))}")
print(f"Citations collected: {len(researcher_result.get('citations', []))}")
print(f"Next agent: {researcher_result.get('current_agent')}")
print(f"Handoffs: {len(researcher_result.get('handoffs', []))}")

2025-11-24 22:20:02 | INFO     | tools.web_search | Executing web search: query='Apache Kafka Fundamentals overview fundamentals'


=== Researcher Node Execution ===


2025-11-24 22:20:03 | INFO     | tools.web_search | Formatting 5 search results
2025-11-24 22:20:03 | INFO     | tools.web_search | Search results formatted: 1432 chars
2025-11-24 22:20:03 | INFO     | tools.web_search | Executing web search: query='Apache Kafka Fundamentals 2024 latest updates'
2025-11-24 22:20:05 | INFO     | tools.web_search | Formatting 5 search results
2025-11-24 22:20:05 | INFO     | tools.web_search | Search results formatted: 1412 chars
2025-11-24 22:20:05 | INFO     | tools.web_search | Executing web search: query='Apache Kafka Fundamentals core concepts'
2025-11-24 22:20:05 | INFO     | tools.web_search | Formatting 5 search results
2025-11-24 22:20:05 | INFO     | tools.web_search | Search results formatted: 1652 chars
Limited research: 1 facts, 1 sources. Proceeding with caution.


Facts gathered: 1
Citations collected: 1
Next agent: writer
Handoffs: 1


In [44]:
# Execute writer node
print("=== Writer Node Execution ===")
writer_result = writer_node(researcher_result)

print(f"Draft created: {'draft.md' in writer_result.get('artifacts', {})}")
if 'draft.md' in writer_result.get('artifacts', {}):
    draft = writer_result['artifacts']['draft.md']
    print(f"Draft length: {len(draft.get('content', ''))} chars")
    print(f"Draft version: {draft.get('version')}")
print(f"Diagram intents: {len(writer_result.get('diagram_intents', []))}")
print(f"Next agent: {writer_result.get('current_agent')}")

=== Writer Node Execution ===
Draft created: True
Draft length: 927 chars
Draft version: 1
Diagram intents: 1
Next agent: editor


In [45]:
# Preview draft content
if 'draft.md' in writer_result.get('artifacts', {}):
    print("=== Draft Preview (first 1000 chars) ===")
    draft_content = writer_result['artifacts']['draft.md']['content']
    print(draft_content[:1000])
    print("...")

=== Draft Preview (first 1000 chars) ===
---
title: Apache Kafka Fundamentals
version: 1
created_by: writer
claim_count: 1
---

# Apache Kafka Fundamentals

**Web Search Results:**

**1. Understanding the Fundamentals of Kafka | by Abdur Razzak**
   Apache Kafka is a distributed, open-source messaging platform designed for high-throughput, low-latency data streaming....
   Source: https://medium.com/@razzak.cse65/understanding-the-fundamentals-of-kafka-
<!-- claim_id: CLAIM_17233E15 -->

## Summary

This article explores apache kafka fundamentals, covering key concepts, 
best practices, and practical applications.

## Key Concepts

## Key Takeaways

- **Web Search Results:**

**1. Understanding the Fundamentals of Kafka | by Abdur Razzak**
   Apache 
  <!-- claim_id: CLAIM_17233E15 -->

## What's Next?

To learn more about apache kafka fundamentals, explore the official documentation 
and try implementing these concepts in your projects.

## References


...


In [46]:
# Execute editor node
print("=== Editor Node Execution ===")
editor_result = editor_node(writer_result)

review = editor_result.get('editor_review', {})
print(f"Review passed: {review.get('pass')}")
print(f"Critical issues: {review.get('critical_count', 0)}")
print(f"Major issues: {review.get('major_count', 0)}")
print(f"Minor issues: {review.get('minor_count', 0)}")
print(f"Iteration: {review.get('iteration')}")
print(f"Next agent: {editor_result.get('current_agent')}")

=== Editor Node Execution ===
Review passed: False
Critical issues: 0
Major issues: 4
Minor issues: 1
Iteration: 1
Next agent: writer


In [47]:
# Show issues if any
issues = editor_result.get('issues', [])
if issues:
    print("=== Issues Found ===")
    for issue in issues[:5]:  # Show first 5
        print(f"\n[{issue.get('severity', 'unknown').upper()}] {issue.get('title', 'No title')}")
        print(f"  Category: {issue.get('category')}")
        print(f"  Description: {issue.get('description')[:100]}...")
else:
    print("No issues found!")

=== Issues Found ===

[MAJOR] Insufficient sources for CLAIM_17233E15
  Category: citation
  Description: Claim CLAIM_17233E15 has only 1 source(s), requires >=2...

[MAJOR] Insufficient sources for CLAIM_17233E15
  Category: citation
  Description: Claim CLAIM_17233E15 has only 1 source(s), requires >=2...

[MAJOR] Low domain diversity
  Category: citation
  Description: Domain  represents 100.0% of sources (>60%)...

[MAJOR] Reading level mismatch
  Category: readability
  Description: FK Grade 0.0 vs target 10...

[MINOR] Sentence length outside range
  Category: readability
  Description: Avg 10.7 words vs target 12-20 words...


In [48]:
# Execute diagrammer node (if there are diagram intents)
if editor_result.get('diagram_intents'):
    print("=== Diagrammer Node Execution ===")
    diagrammer_result = diagrammer_node(editor_result)
    
    print(f"Diagrams generated: {len(diagrammer_result.get('diagrams', []))}")
    print(f"Workflow complete: {diagrammer_result.get('workflow_complete')}")
    
    for diagram in diagrammer_result.get('diagrams', []):
        print(f"\nDiagram: {diagram.get('title')}")
        print(f"  Type: {diagram.get('type')}")
        print(f"  Nodes: {diagram.get('node_count')}")
        print(f"  Edges: {diagram.get('edge_count')}")
else:
    print("No diagram intents - skipping diagrammer")

=== Diagrammer Node Execution ===
Diagrams generated: 1
Workflow complete: True

Diagram: Apache Kafka Fundamentals Architecture Overview
  Type: architecture
  Nodes: 4
  Edges: 3


## 7. Full Middleware Chain

Demonstrate the complete middleware chain.

In [49]:
# Get the middleware chain
chain = get_middleware_chain()

print("=== Middleware Chain ===")
print(f"Total handlers: {len(chain.handlers)}")
for i, handler in enumerate(chain.handlers, 1):
    print(f"  {i}. {handler.__class__.__name__}")

=== Middleware Chain ===
Total handlers: 8
  1. LimitsHandler
  2. PIIHandler
  3. HITLHandler
  4. RetryHandler
  5. FallbackHandler
  6. SummarizationHandler
  7. StyleInjectorHandler
  8. CitationNormalizerHandler


In [50]:
# Test middleware chain processing
test_state = {
    "run_id": "test_run",
    "model_calls": 0,
    "tool_calls": 0,
    "content": "Test content with email: test@example.com",
}

print("=== Pre-processing ===")
pre_result = chain.process_request(test_state, "researcher")
print(f"Limits checked: {pre_result.get('_limits_checked', False)}")

print("\n=== Post-processing ===")
post_result = chain.process_response(test_state, "writer")
print(f"PII redacted: {'[EMAIL_REDACTED]' in post_result.get('content', '')}")

=== Pre-processing ===
Limits checked: True

=== Post-processing ===
PII redacted: True


## 8. End-to-End Workflow Demo

Run the complete newsletter workflow step by step to see all agents in action.

In [51]:
# Run the complete workflow step by step
print("=" * 60)
print("RUNNING COMPLETE WORKFLOW")
print("=" * 60)

# Create initial state
initial_state = create_initial_state(
    topic="Apache Kafka for Data Engineers",
    brief="A comprehensive guide to Kafka for data engineering teams",
    requirements=[
        "Explain core concepts",
        "Cover performance tuning",
        "Include real-world use cases",
    ],
    style_profile=style_profile,
    max_iterations=3,
)

print(f"\n[INITIAL STATE]")
print(f"  Topic: {initial_state['tasks'][0]['topic']}")
print(f"  Run ID: {initial_state['run_id']}")

# Step 1: Researcher
print(f"\n{'='*60}")
print("[STEP 1: RESEARCHER]")
print("="*60)
state_after_researcher = researcher_node(initial_state)
print(f"  Facts gathered: {len(state_after_researcher.get('facts', []))}")
print(f"  Citations collected: {len(state_after_researcher.get('citations', []))}")
print(f"  Handoff to: {state_after_researcher.get('current_agent')}")

if state_after_researcher.get('facts'):
    print(f"\n  Sample Facts:")
    for fact in state_after_researcher.get('facts', [])[:3]:
        print(f"    - [{fact.get('claim_id')}] {fact.get('text', '')[:80]}...")

# Step 2: Writer
print(f"\n{'='*60}")
print("[STEP 2: WRITER]")
print("="*60)
state_after_writer = writer_node(state_after_researcher)
print(f"  Draft created: {'draft.md' in state_after_writer.get('artifacts', {})}")
if 'draft.md' in state_after_writer.get('artifacts', {}):
    draft = state_after_writer['artifacts']['draft.md']
    print(f"  Draft length: {len(draft.get('content', ''))} characters")
    print(f"  Draft version: {draft.get('version')}")
print(f"  Diagram intents: {len(state_after_writer.get('diagram_intents', []))}")
print(f"  Handoff to: {state_after_writer.get('current_agent')}")

# Step 3: Editor
print(f"\n{'='*60}")
print("[STEP 3: EDITOR]")
print("="*60)
state_after_editor = editor_node(state_after_writer)
review = state_after_editor.get('editor_review', {})
print(f"  Review PASSED: {review.get('pass')}")
print(f"  Critical issues: {review.get('critical_count', 0)}")
print(f"  Major issues: {review.get('major_count', 0)}")
print(f"  Minor issues: {review.get('minor_count', 0)}")
print(f"  Iteration: {review.get('iteration')}")
print(f"  Handoff to: {state_after_editor.get('current_agent', 'END')}")

if state_after_editor.get('issues'):
    print(f"\n  Issues Found:")
    for issue in state_after_editor.get('issues', [])[:5]:
        print(f"    - [{issue.get('severity', '?').upper()}] {issue.get('title', 'No title')}")

# Step 4: Diagrammer (if there are diagram intents)
if state_after_editor.get('diagram_intents') and state_after_editor.get('current_agent') == 'diagrammer':
    print(f"\n{'='*60}")
    print("[STEP 4: DIAGRAMMER]")
    print("="*60)
    state_final = diagrammer_node(state_after_editor)
    print(f"  Diagrams generated: {len(state_final.get('diagrams', []))}")
    for diagram in state_final.get('diagrams', []):
        print(f"    - {diagram.get('title')} ({diagram.get('type')})")
        print(f"      Nodes: {diagram.get('node_count')}, Edges: {diagram.get('edge_count')}")
    print(f"  Workflow complete: {state_final.get('workflow_complete')}")
else:
    state_final = state_after_editor
    print(f"\n[STEP 4: DIAGRAMMER - SKIPPED]")
    print(f"  No diagram intents or editor did not pass to diagrammer")

# Final Summary
print(f"\n{'='*60}")
print("WORKFLOW COMPLETE - SUMMARY")
print("="*60)
print(f"  Total Facts: {len(state_final.get('facts', []))}")
print(f"  Total Citations: {len(state_final.get('citations', []))}")
print(f"  Draft Created: {'draft.md' in state_final.get('artifacts', {})}")
print(f"  Diagrams Created: {len(state_final.get('diagrams', []))}")
print(f"  Editor Pass: {state_final.get('editor_review', {}).get('pass', False)}")
print(f"  Total Iterations: {state_final.get('iteration_count', 0)}")
print(f"  Handoff History: {len(state_final.get('handoffs', []))} handoffs")

2025-11-24 22:20:05 | INFO     | tools.web_search | Executing web search: query='Apache Kafka for Data Engineers overview fundamentals'


RUNNING COMPLETE WORKFLOW

[INITIAL STATE]
  Topic: Apache Kafka for Data Engineers
  Run ID: run_e4c1b1a478da

[STEP 1: RESEARCHER]


2025-11-24 22:20:07 | INFO     | tools.web_search | Formatting 5 search results
2025-11-24 22:20:07 | INFO     | tools.web_search | Search results formatted: 2007 chars
2025-11-24 22:20:07 | INFO     | tools.web_search | Executing web search: query='Apache Kafka for Data Engineers 2024 latest updates'
2025-11-24 22:20:08 | INFO     | tools.web_search | Formatting 5 search results
2025-11-24 22:20:08 | INFO     | tools.web_search | Search results formatted: 1579 chars
2025-11-24 22:20:08 | INFO     | tools.web_search | Executing web search: query='Apache Kafka for Data Engineers Explain core concepts'
2025-11-24 22:20:10 | INFO     | tools.web_search | Formatting 5 search results
2025-11-24 22:20:10 | INFO     | tools.web_search | Search results formatted: 1912 chars
Limited research: 1 facts, 1 sources. Proceeding with caution.


  Facts gathered: 1
  Citations collected: 1
  Handoff to: writer

  Sample Facts:
    - [CLAIM_2E90355E] **Web Search Results:**

**1. Apache Kafka Fundamentals - Learn Data Engineering...

[STEP 2: WRITER]
  Draft created: True
  Draft length: 951 characters
  Draft version: 1
  Diagram intents: 1
  Handoff to: editor

[STEP 3: EDITOR]
  Review PASSED: False
  Critical issues: 0
  Major issues: 4
  Minor issues: 0
  Iteration: 1
  Handoff to: writer

  Issues Found:
    - [MAJOR] Insufficient sources for CLAIM_2E90355E
    - [MAJOR] Insufficient sources for CLAIM_2E90355E
    - [MAJOR] Low domain diversity
    - [MAJOR] Reading level mismatch

[STEP 4: DIAGRAMMER - SKIPPED]
  No diagram intents or editor did not pass to diagrammer

WORKFLOW COMPLETE - SUMMARY
  Total Facts: 1
  Total Citations: 1
  Draft Created: True
  Diagrams Created: 0
  Editor Pass: False
  Total Iterations: 1
  Handoff History: 3 handoffs


In [52]:
# Show the actual generated content
print("=" * 60)
print("GENERATED CONTENT PREVIEW")
print("=" * 60)

# Show draft preview
if 'draft.md' in state_final.get('artifacts', {}):
    print("\n[DRAFT PREVIEW - First 1500 characters]")
    print("-" * 40)
    draft_content = state_final['artifacts']['draft.md']['content']
    print(draft_content[:1500])
    if len(draft_content) > 1500:
        print("\n... (truncated)")

# Show diagram preview
if state_final.get('diagrams'):
    print("\n\n[DIAGRAM XML PREVIEW]")
    print("-" * 40)
    for diagram in state_final.get('diagrams', []):
        print(f"Diagram: {diagram.get('title')}")
        drawio_xml = diagram.get('drawio_xml', '')
        print(f"DrawIO XML ({len(drawio_xml)} characters):")
        # Show first 500 chars of XML
        print(drawio_xml[:500])
        if len(drawio_xml) > 500:
            print("\n... (truncated)")

# Show all citations
if state_final.get('citations'):
    print("\n\n[CITATIONS COLLECTED]")
    print("-" * 40)
    for i, citation in enumerate(state_final.get('citations', []), 1):
        print(f"{i}. [{citation.get('source_id', 'N/A')}] {citation.get('title', 'No title')[:60]}...")
        print(f"   URL: {citation.get('url', 'N/A')[:80]}")

GENERATED CONTENT PREVIEW

[DRAFT PREVIEW - First 1500 characters]
----------------------------------------
---
title: Apache Kafka for Data Engineers
version: 1
created_by: writer
claim_count: 1
---

# Apache Kafka for Data Engineers

**Web Search Results:**

**1. Apache Kafka Fundamentals - Learn Data Engineering Academy**
   In this training you learn all the basics you need to start working with Apache Kafka. Learn how you can set up your queue and how to write message producers...
   Source: https://learndataengineering.com/p
<!-- claim_id: CLAIM_2E90355E -->

## Summary

This article explores apache kafka for data engineers, covering key concepts, 
best practices, and practical applications.

## Key Concepts

## Key Takeaways

- **Web Search Results:**

**1. Apache Kafka Fundamentals - Learn Data Engineering Academy**
   In thi
  <!-- claim_id: CLAIM_2E90355E -->

## What's Next?

To learn more about apache kafka for data engineers, explore the official documentation 
and try imp

In [53]:
# Create the Newsletter Workflow to verify graph compilation
workflow = NewsletterWorkflow()

print("=== Newsletter Workflow ===")
print(f"Graph created: {workflow.graph is not None}")
print(f"\nWorkflow Flow:")
print("  1. Researcher -> Gathers facts and citations")
print("  2. Writer -> Creates draft from facts") 
print("  3. Editor -> Reviews quality, pass/fail")
print("  4. Diagrammer -> Generates diagrams (if needed)")
print("  5. END -> Workflow complete")

=== Newsletter Workflow ===
Graph created: True

Workflow Flow:
  1. Researcher -> Gathers facts and citations
  2. Writer -> Creates draft from facts
  3. Editor -> Reviews quality, pass/fail
  4. Diagrammer -> Generates diagrams (if needed)
  5. END -> Workflow complete


## 9. Phase 2 Pass Criteria Verification

Verify that Phase 2 pass criteria are met.

In [54]:
print("=== Phase 2 Pass Criteria Verification ===")

criteria = [
    ("Middleware chain created", len(chain.handlers) >= 5),
    ("Tool binding per department", all(
        tool_binder.get_allowed_tools(d) for d in departments
    )),
    ("HITL simulator functional", auto_approve.process(request_trusted)['decision'] == 'approve'),
    ("State initialization works", demo_state['run_id'].startswith('run_')),
    ("Researcher node executable", 'current_agent' in researcher_result),
    ("Writer produces draft", 'draft.md' in writer_result.get('artifacts', {})),
    ("Editor produces review", 'editor_review' in editor_result),
    ("Workflow graph compiles", workflow.graph is not None),
]

all_passed = True
for criterion, passed in criteria:
    status = "PASS" if passed else "FAIL"
    symbol = "" if passed else ""
    print(f"{symbol} {criterion}: {status}")
    if not passed:
        all_passed = False

print(f"\n{'=' * 40}")
print(f"OVERALL: {'ALL CRITERIA PASSED!' if all_passed else 'SOME CRITERIA FAILED'}")

=== Phase 2 Pass Criteria Verification ===
 Middleware chain created: PASS
 Tool binding per department: PASS
 HITL simulator functional: PASS
 State initialization works: PASS
 Researcher node executable: PASS
 Writer produces draft: PASS
 Editor produces review: PASS
 Workflow graph compiles: PASS

OVERALL: ALL CRITERIA PASSED!


## 10. Summary

Phase 2 implementation is complete with:

1. **Middleware Stack** (8 layers):
   - Limits enforcement
   - PII redaction
   - HITL gating
   - Retry logic
   - Model fallback
   - Summarization
   - Style injection
   - Citation normalization

2. **Tool Binding**:
   - Researcher: web.search, web.fetch, local.search, citation.builder
   - Writer: seo.readability
   - Editor: seo.readability, plagiarism.scan, citation.builder
   - Diagrammer: drawio.generate, drawio.export, web.search

3. **HITL Simulation**:
   - AUTO_APPROVE mode
   - AUTO_DENY mode
   - POLICY mode (domain-based)
   - INTERACTIVE mode (for UI integration)

4. **Agent Nodes**:
   - ResearcherNode: Gathers facts and citations
   - WriterNode: Transforms research into draft
   - EditorNode: Reviews and enforces quality
   - DiagrammerNode: Generates DrawIO diagrams

5. **LangGraph Workflow**:
   - Entry: Researcher
   - Flow: Researcher -> Writer -> Editor -> Diagrammer/END
   - Conditional routing based on editor review
   - Stop conditions: draft.md + editor_pass or max_iterations